In [2]:
import json
import os
from datetime import datetime, timedelta
from typing import Dict, Tuple

import pandas as pd


In [ ]:
# Dataset configuration
DATASETS = ["Books", "Electronics", "Sports", "Tools"]
DATASET_NAME = "".join([d[0] for d in DATASETS])
OUTPUT_DIR = f"path/to/ABXI/data/{DATASET_NAME}"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [9]:
def load_item_maps(dataset_name: str) -> Dict[str, list]:
    """
    Load item mappings from the dataset .item file.
    Each item is mapped to its index and domain ID.
    """
    item_path = f"./{dataset_name}/{dataset_name}.item"
    if not os.path.exists(item_path):
        raise FileNotFoundError(f"Item file not found at {item_path}")

    item_maps = {}
    with open(item_path, "r") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            dom = ord(line.split("_")[-1]) - ord("A")
            item_maps[line] = [i + 1, dom]
    return item_maps


In [10]:
def process_sequences(dataset_name: str) -> Tuple[Dict[str, str], pd.DataFrame]:
    """
    Process user interaction sequences and attach timestamps.
    Returns:
        user_maps: mapping from user_id to user_id (identity map for now).
        seq_df: DataFrame with users and their timestamped sequences.
    """
    inter_path = f"./{dataset_name}/{dataset_name}.full.test.inter"
    if not os.path.exists(inter_path):
        raise FileNotFoundError(f"Interaction file not found at {inter_path}")

    df = pd.read_csv(
        inter_path,
        sep="\t",
        header=0,
        names=["user_id", "item_seq", "target_item"],
        dtype={"user_id": str, "item_seq": str, "target_item": str},
    )

    base_time = datetime(2025, 1, 1, 0, 0, 0)
    time_interval = timedelta(hours=1)

    users, seqs, user_maps = [], [], {}
    for _, row in df.iterrows():
        user = row["user_id"]
        item_seq = row["item_seq"].split() + [row["target_item"]]

        seq = " ".join(
            f"{item}|{int((base_time + i * time_interval).timestamp())}"
            for i, item in enumerate(item_seq)
        )

        users.append(user)
        seqs.append(seq)
        user_maps[user] = int(user)

    return user_maps, pd.DataFrame({"users": users, "seqs": seqs})



In [11]:
def save_dataframes(
    seq_df: pd.DataFrame,
    user_maps: Dict[str, str],
    item_maps: Dict[str, list],
    output_dir: str,
    dataset_name: str,
) -> None:
    """
    Save preprocessed sequences and mappings to files.
    """
    # Save sequence data
    seq_path = os.path.join(output_dir, f"{dataset_name}_128_preprocessed.txt")
    with open(seq_path, "w") as f:
        for user, seq in zip(seq_df["users"], seq_df["seqs"]):
            f.write(f"{user} {seq}\n")

    # Save user and item mappings in JSON format
    user_path = os.path.join(output_dir, "map_user_128.txt")
    item_path = os.path.join(output_dir, "map_item_128.txt")
    with open(user_path, "w") as f:
        json.dump(user_maps, f, separators=(", ", ": "))
    with open(item_path, "w") as f:
        json.dump(item_maps, f, separators=(", ", ": "))


In [12]:
try:
    item_maps = load_item_maps(DATASET_NAME)
    user_maps, seq_df = process_sequences(DATASET_NAME)
    save_dataframes(seq_df, user_maps, item_maps, OUTPUT_DIR, DATASET_NAME)
    print("\n✅ Data processing complete.")
except Exception as e:
    print(f"\n❌ Error: {e}")



✅ Data processing complete.
